# Lesson 2: Tools with real stock data
A real user says *"Should I buy Apple?"*, not *"AAPL"*. So the agent needs a tool that finds the ticker from a company name, then more tools that use that ticker.

Run the cells top to bottom with **Shift+Enter**. Notes are in `langchain.md`.

## Step 0: Setup

In [1]:
from dotenv import load_dotenv
import yfinance as yf
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()
MODEL = 'groq:openai/gpt-oss-120b'

## Step 1: Tool that finds the ticker from a company name
The docstring says *when* to use it. This tool runs first, and its result feeds the other tools.

In [2]:
@tool
def search_ticker(company_name: str) -> str:
    """Find the stock ticker symbol for a company name like 'Apple' or 'Tesla'.
    Always use this first when the user gives a company name instead of a ticker."""
    results = yf.Search(company_name, max_results=6, news_count=0).quotes
    stocks = [q for q in results if q.get('quoteType') == 'EQUITY']
    if not stocks:
        return f"No stock found for '{company_name}'."
    return "\n".join(f"{q['symbol']}: {q.get('shortname')} ({q.get('exchange')})" for q in stocks)

print(search_ticker.invoke({'company_name': 'Apple'}))

AAPL: Apple Inc. (NMS)
APLE: Apple Hospitality REIT, Inc. (NYQ)


## Step 2: Tool for current price and key numbers

In [3]:
@tool
def get_snapshot(ticker: str) -> str:
    """Get current price and key numbers for a stock ticker like AAPL:
    price, P/E ratio, market cap, sector, 52-week range."""
    info = yf.Ticker(ticker).info
    keys = ['longName', 'sector', 'currentPrice', 'currency', 'trailingPE', 'forwardPE',
            'marketCap', 'fiftyTwoWeekLow', 'fiftyTwoWeekHigh', 'recommendationKey']
    return "\n".join(f"{k}: {info.get(k)}" for k in keys)

print(get_snapshot.invoke({'ticker': 'AAPL'}))

longName: Apple Inc.
sector: Technology
currentPrice: 354.07
currency: USD
trailingPE: 38.866085
forwardPE: 33.488987
marketCap: 5145409617920
fiftyTwoWeekLow: 243.42
fiftyTwoWeekHigh: 360.83
recommendationKey: buy


## Step 3: Tool for the price trend
We summarize the history into a few numbers instead of sending hundreds of rows. Sending less data is cheaper and easier for the model to read.

In [4]:
@tool
def get_price_trend(ticker: str, period: str = '6mo') -> str:
    """Get how a stock's price changed over a period. Period is one of:
    1mo, 3mo, 6mo, 1y, 5y. Returns start, end, change in percent, high and low."""
    close = yf.Ticker(ticker).history(period=period)['Close']
    if close.empty:
        return f'No price data for {ticker}.'
    start, end = close.iloc[0], close.iloc[-1]
    return (f"{ticker} over {period}: start ${start:.2f}, end ${end:.2f}, "
            f"change {100 * (end - start) / start:+.1f}%, high ${close.max():.2f}, low ${close.min():.2f}")

print(get_price_trend.invoke({'ticker': 'AAPL', 'period': '6mo'}))

AAPL over 6mo: start $307.83, end $354.07, change +15.0%, high $356.27, low $274.70


## Step 4: Tool for recent news
News is the evidence for *why* a stock is moving.

In [5]:
@tool
def get_recent_news(ticker: str) -> str:
    """Get the latest news headlines for a stock ticker like AAPL."""
    lines = []
    for item in yf.Ticker(ticker).news[:5]:
        c = item.get('content', item)
        lines.append(f"- {c.get('title')} ({c.get('pubDate', '')[:10]})")
    return "\n".join(lines) or 'No recent news.'

print(get_recent_news.invoke({'ticker': 'AAPL'}))

- Why the Jensen Huang rally faded so quickly (2026-11-24)
- Stocks give back gains as Anthropic dips below IPO price (2026-11-24)
- Who is holding Jensen Huang accountable? It sure looks like nobody (2026-11-24)
- Xiaomi shares jump over 8% as premium phone sales cushion rising chip costs (2026-11-25)
- 5 Sleeper Blue-Chip Stock Picks for Steady Long-Term Gains (2026-11-25)


## Step 4b: Tool to compare against the whole market
Without this, the agent could not honestly say a stock "beat the market", so it made the claim up. `^GSPC` is the S&P 500, a standard measure of the US market.

In [10]:
@tool
def get_market_trend(period: str = '6mo') -> str:
    """Get how the whole US market (S&P 500) changed over a period. Use it to compare a stock
    against the market. Period is one of: 1mo, 3mo, 6mo, 1y, 5y. Use the same period as the stock."""
    close = yf.Ticker('^GSPC').history(period=period)['Close']
    if close.empty:
        return 'No market data.'
    start, end = close.iloc[0], close.iloc[-1]
    return f"S&P 500 over {period}: change {100 * (end - start) / start:+.1f}%"

print(get_market_trend.invoke({'period': '6mo'}))

S&P 500 over 6mo: change -4.2%


## Step 5: Give all the tools to the agent
The system prompt now has strict rules. We never tell the agent which tool to call. It decides.

In [11]:
agent = create_agent(
    model=MODEL,
    tools=[search_ticker, get_snapshot, get_price_trend, get_market_trend, get_recent_news],
    system_prompt=(
        "You are a careful stock research assistant. The user gives company names, not tickers. "
        "First find the ticker, then check the snapshot, the price trend, the market trend "
        "over the same period, and the news.\n"
        "Rules:\n"
        "1. Only state numbers that came from a tool result in this conversation.\n"
        "2. Only call something 'above' or 'below' the market if you have both numbers.\n"
        "3. Only mention news that is actually about the company. Ignore unrelated headlines and say so.\n"
        "4. If you lack data for a point, say 'I don't have data on this' instead of guessing.\n"
        "5. Say clearly that this is research, not financial advice."
    ),
)

result = agent.invoke({'messages': [{'role': 'user', 'content': 'I want to look into Apple stock. Is it doing well?'}]})
print(result['messages'][-1].content)

**Apple Inc. (Ticker: AAPL) – Research Summary**

| Item | Value (from tool data) |
|------|------------------------|
| **Current price** | **$354.07** |
| **Trailing P/E** | **38.87** |
| **Forward P/E** | **33.49** |
| **Market cap** | **$5.15 trillion** |
| **Sector** | Technology |
| **52‑week range** | **$243.42 – $360.83** |
| **6‑month price change** | **+15.0 %** (from $307.83 to $354.07) |
| **6‑month high / low** | **$356.27 / $274.70** |
| **S&P 500 6‑month change** | **‑4.2 %** |

**Interpretation (research, not advice)**  
- Over the past six months Apple’s share price rose 15 % while the broader S&P 500 fell 4.2 %. This indicates Apple has *outperformed* the overall market in that period.  
- The stock is trading near the top of its 52‑week range (just below the high of $360.83).  
- A trailing P/E of ~38.9 is higher than the market average, suggesting the market is pricing in strong growth expectations; the forward P/E is a bit lower (~33.5), indicating analysts expect e

## Step 6: Watch which tools it called, in order

In [7]:
for m in result['messages']:
    if m.type == 'ai' and m.tool_calls:
        for call in m.tool_calls:
            print('CALL:', call['name'], call['args'])

CALL: search_ticker {'company_name': 'Apple'}
CALL: get_snapshot {'ticker': 'AAPL'}
CALL: get_price_trend {'period': '6mo', 'ticker': 'AAPL'}
CALL: get_recent_news {'ticker': 'AAPL'}


## Step 7: Check which models your key can use
Run this if you ever get a `model_not_found` error.

In [8]:
import os, json, urllib.request

req = urllib.request.Request('https://api.groq.com/openai/v1/models',
    headers={'Authorization': f"Bearer {os.environ['GROQ_API_KEY']}", 'User-Agent': 'python'})
for m in sorted(json.load(urllib.request.urlopen(req))['data'], key=lambda m: m['id']):
    print(m['id'])

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo


## Exercise
1. Ask about a company with a tricky name, like *"Meta"* or *"Google"*. Which ticker did the agent pick, and why?
2. Ask for *two* companies at once, like *"Compare Nvidia and AMD"*. Check Step 6 to see how many tool calls it made.
3. Add a tool `get_analyst_opinion(ticker)` that returns `recommendationKey` and `targetMeanPrice` from `yf.Ticker(ticker).info`.

In [ ]:
# your code here